In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler,StandardScaler,LabelEncoder


data=pd.read_csv("/content/Airbnb_Open_Data.csv",low_memory=False)
df=pd.DataFrame(data)

df.columns = df.columns.str.lower().str.replace(" ", "_")

df["name"].fillna("unknown",inplace=True)
df["host_name"].fillna("unknown",inplace=True)
df["host_identity_verified"].fillna("unknown",inplace=True)
df["neighbourhood"].fillna("unknown",inplace=True)
df["neighbourhood_group"].fillna("unknown",inplace=True)
df["calculated_host_listings_count"].fillna(df['calculated_host_listings_count'].median(),inplace=True)
df = df.dropna(subset=["latitude", "longitude"])
df.drop("country_code", axis=1, inplace=True)
df["instant_bookable"].fillna("unknown", inplace=True)
df = pd.get_dummies(df, columns=["instant_bookable"])
df["construction_year"].fillna(df['construction_year'].median(),inplace=True)
df = df.dropna(subset=["price"])
df = df.dropna(subset=["service_fee"])
df["minimum_nights"].fillna(df["minimum_nights"].median(), inplace=True)
df = df.drop(columns=["last_review_date"],errors="ignore")
df["reviews_per_month"].fillna(0, inplace=True)
df["review_rate_number"].fillna(df["review_rate_number"].median(), inplace=True)
df["availability_365"].fillna(df["availability_365"].median(), inplace=True)
df['house_rules'].fillna('none', inplace=True)
df=df.drop(columns=["license"],errors="ignore")

df["price"] = df["price"].replace('[\$,]', '', regex=True)
df["price"] = pd.to_numeric(df["price"], errors="coerce")

df["service_fee"] = df["service_fee"].replace('[\$,]', '', regex=True)
df["service_fee"] = pd.to_numeric(df["service_fee"], errors="coerce")

df=df.dropna(subset=["price","service_fee"])

def remove_outliers(df,column):
  Q1=np.quantile(df[column],0.25)
  Q3=np.quantile(df[column],0.75)
  IQR=Q3-Q1

  lower_bound=Q1-IQR*1.5
  upper_bound=Q3+IQR*1.5
  return df[(df[column]>=lower_bound) & (df[column]<=upper_bound)]

df = remove_outliers(df, "price")
df = remove_outliers(df, "service_fee")
df = remove_outliers(df, "minimum_nights")

df=pd.get_dummies(df,column=["host_identity_verified"],drop_first=True)

mapping = {
    "flexible": 0,
    "moderate": 1,
    "strict": 2
}

df["cancellation_policy"] = df["cancellation_policy"].map(mapping)


df["price_per_night"] = df["price"] / df["minimum_nights"]

df["availability_level"] = pd.cut(
    df["availability_365"],
    bins=[0, 50, 200, 365],
    labels=["low", "medium", "high"]
)
df["is_highly_rated"] = df["review_rate_number"] >= 4.5

df["is_active"] = df["reviews_per_month"] > 0

df["host_experience"] = pd.cut(
    df["calculated_host_listings_count"],
    bins=[0, 1, 5, 20, 100],
    labels=["new", "small", "experienced", "professional"]
)
df["is_expensive"] = df["price"] > df["price"].median()

df["price_category"] = pd.cut(
    df["price"],
    bins=3,
    labels=["low", "medium", "high"]
)

/tmp/ipykernel_20351/665811129.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["NAME"].fillna("unknown",inplace=True)
/tmp/ipykernel_20351/665811129.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using

<bound method DataFrame.dropna of              lat      long
0       40.64749 -73.97237
1       40.75362 -73.98377
2       40.80902 -73.94190
3       40.68514 -73.95976
4       40.79851 -73.94399
...          ...       ...
102594  40.70862 -73.94651
102595  40.80460 -73.96545
102596  40.67505 -73.98045
102597  40.74989 -73.93777
102598  40.76807 -73.98342

[102599 rows x 2 columns]>